# Description
We demonstrate how to extract a concept vector for a concept using the method outlined by Lindsey et. al. in their ["Emergent Introspective Awareness"](https://transformer-circuits.pub/2025/introspection/index.html) paper. This is for Llama-8.1-3B-Instruct, but should work with minor modifications for any model in the Llama-3.1 family.

We define a few global configuration variables, note that you'll need a `.env` file with the values that we retrieve from the environment in the below cell.

In [46]:
from dotenv import load_dotenv
import os

MODEL="meta-llama/Llama-3.1-70B-Instruct"
NDIF_API_KEY=os.environ["NDIF_API_KEY"]


Begin with a health check on the NDIF API, which is used throughout this notebook.

In [47]:
import nnsight
from pprint import pprint

assert nnsight.is_model_running(MODEL), f"{MODEL} is not online."

status = nnsight.ndif_status()

print(status)

target = MODEL

NDIF Service: Up 🟢 

┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Model Class   ┃ Repo ID                            ┃ Revision ┃ Type       ┃ Status       ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ LanguageModel │ meta-llama/Llama-3.1-8B-Instruct   │ main     │ Pilot-Only │ RUNNING      │
│ LanguageModel │ google/gemma-2-9b-it               │ main     │ Dedicated  │ RUNNING      │
│ LanguageModel │ meta-llama/Llama-3.1-8B            │ main     │ Dedicated  │ RUNNING      │
│ LanguageModel │ google/gemma-3-12b-pt              │ main     │ Pilot-Only │ RUNNING      │
│ LanguageModel │ meta-llama/Llama-3.1-70B-Instruct  │ main     │ Dedicated  │ RUNNING      │
│ LanguageModel │ meta-llama/Llama-3.3-70B-Instruct  │ main     │ Dedicated  │ RUNNING      │
│ LanguageModel │ meta-llama/Llama-3.1-70B           │ main     │ Dedicated  │ RUNNING      │
│ LanguageModel │ meta-llama/Llama-3.1-405B-Instruct │ main     │ Dedicated  │ RUNNING      │
│ LanguageModel │ openai-community/gpt2              │ main     │ Dedicated  │ RUNNING      │
│ LanguageModel │ EleutherAI/gpt-j-6b                │ main     │ Dedicated  │ RUNNING      │
│ LanguageModel │ meta-llama/Llama-3.1-405B          │ main     │ Scheduled  │ NOT DEPLOYED │
└───────────────┴────────────────────────────────────┴──────────┴────────────┴──────────────┘

Load the model and perform a quick test.

In [48]:
import nnsight

model = nnsight.LanguageModel(MODEL)

prompt = "Reply with exactly three words:"

with model.generate(prompt, max_new_tokens=10, do_sample=False, remote=True):
    out_ids = model.generator.output.save()

print(out_ids)

text = model.tokenizer.decode(out_ids, skip_special_tokens=True)
print(text)

⬇ Downloading:   0%|          | 0.00/1.77k [00:00<?]

tensor([[128000,  21509,    449,   7041,   2380,   4339,     25,    358,   1097,
           1618,    627,     40,   1097,   1618,     13, 128009]])
['Reply with exactly three words: I am here.\nI am here.']


Test if we can extract the activations of the model at each layer.

In [98]:
word = "recursion"
messages = [{"role": "user", "content": f"Tell me about {word}."}]
prompt = model.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

# demonstration of how to extract activations from each layer
with model.trace(prompt, remote=True) as tracer:
    cache = tracer.cache(
        modules=[layer for layer in model.model.layers]
    ).save()

# Access after the trace
print(cache["model.model.layers.0"])

⬇ Downloading:   0%|          | 0.00/53.8M [00:00<?]

{'output': tensor([[[ 0.0009,  0.0050, -0.0017,  ...,  0.0028,  0.0090,  0.0071],
         [ 0.0009,  0.0050, -0.0017,  ...,  0.0028,  0.0090,  0.0071],
         [ 0.0002, -0.0066,  0.0022,  ...,  0.0112, -0.0145, -0.0028],
         ...,
         [ 0.0036,  0.0287, -0.0132,  ..., -0.0026,  0.0009,  0.0067],
         [-0.0003, -0.0005,  0.0019,  ...,  0.0033, -0.0031, -0.0018],
         [ 0.0064,  0.0024,  0.0024,  ...,  0.0016, -0.0003,  0.0012]]],
       dtype=torch.bfloat16), 'inputs': None}


Compute mean baseline last token activation vector using the 100 words in Lindsey et al.

In [101]:
import torch

# Lindsey et al. baseline words
words = "Desks, Jackets, Gondolas, Laughter, Intelligence, Bicycles, Chairs, Orchestras, Sand, Pottery, Arrowheads, Jewelry, Daffodils, Plateaus, Estuaries, Quilts, Moments, Bamboo, Ravines, Archives, Hieroglyphs, Stars, Clay, Fossils, Wildlife, Flour, Traffic, Bubbles, Honey, Geodes, Magnets, Ribbons, Zigzags, Puzzles, Tornadoes, Anthills, Galaxies, Poverty, Diamonds, Universes, Vinegar, Nebulae, Knowledge, Marble, Fog, Rivers, Scrolls, Silhouettes, Marbles, Cakes, Valleys, Whispers, Pendulums, Towers, Tables, Glaciers, Whirlpools, Jungles, Wool, Anger, Ramparts, Flowers, Research, Hammers, Clouds, Justice, Dogs, Butterflies, Needles, Fortresses, Bonfires, Skyscrapers, Caravans, Patience, Bacon, Velocities, Smoke, Electricity, Sunsets, Anchors, Parchments, Courage, Statues, Oxygen, Time, Butterflies, Fabric, Pasta, Snowflakes, Mountains, Echoes, Pianos, Sanctuaries, Abysses, Air, Dewdrops, Gardens, Literature, Rice, Enigmas"
baseline_words = [w.strip() for w in words.split(",") if w.strip()]

num_layers = model.config.num_hidden_layers
baseline_samples = []

# you could test it with the first five samples to reduce computation time
for w in baseline_words:
    messages = [{"role": "user", "content": f"Tell me about {w}."}]
    prompt = model.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    with model.trace(prompt, remote=True) as tracer:
        cache = tracer.cache(
            modules=[layer for layer in model.model.layers]
        ).save()
    
    sample = torch.stack([
        cache[f'model.model.layers.{i}']["output"][0, -1, :]
        for i in range(num_layers)
    ], dim=0)  # [num_layers, hidden]

    baseline_samples.append(sample)

baseline_stack = torch.stack(baseline_samples, dim=0)
baseline_mean = baseline_stack.mean(dim=0)
baseline_mean_by_layer = {i: baseline_mean[i] for i in range(num_layers)}

print("baseline_mean:", baseline_mean.shape)
print("layer 0 vector:", baseline_mean_by_layer[0].shape)


⬇ Downloading:   0%|          | 0.00/55.1M [00:00<?]

⬇ Downloading:   0%|          | 0.00/53.8M [00:00<?]

⬇ Downloading:   0%|          | 0.00/56.4M [00:00<?]

⬇ Downloading:   0%|          | 0.00/55.1M [00:00<?]

⬇ Downloading:   0%|          | 0.00/53.8M [00:00<?]

baseline_mean: torch.Size([80, 8192])
layer 0 vector: torch.Size([8192])


See if we can compute a steering vector for a concept using the previously computed mean baseline vector.

In [102]:
concept = "Dust"

messages = [{"role": "user", "content": f"Tell me about {concept}."}]
prompt = model.tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

# demonstration of how to extract activations from each layer
with model.trace(prompt, remote=True) as tracer:
    cache = tracer.cache(
        modules=[layer for layer in model.model.layers]
    ).save()

sample = torch.stack([
    cache[f'model.model.layers.{i}']["output"][0, -1, :]
    for i in range(num_layers)
], dim=0)

concept_vector_by_layer = {
    idx: sample[idx] - baseline_mean_by_layer[idx] for idx in range(num_layers)
}

⬇ Downloading:   0%|          | 0.00/53.8M [00:00<?]